# Semana 24 — Experiment Tracking e Model Registry com MLflow
**Fase 5: MLOps, Deployment e Agent-First Products** · Gabarito (notebook resolvido)

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NetoRibeiro/ai-architect-foundations/blob/main/05-Phase-5-MLOps-and-Agents/notebooks/semana_24_gabarito.ipynb)

Leitura de conceito desta semana: [Semana-24-Conceitos.md](https://github.com/NetoRibeiro/ai-architect-foundations/blob/main/05-Phase-5-MLOps-and-Agents/docs/Semana-24-Conceitos.md)

Use este gabarito **depois** de tentar o notebook de exercícios. Compare a sua solução, não copie.

In [ ]:
!pip -q install mlflow

In [ ]:
import mlflow, hashlib, numpy as np, pandas as pd, matplotlib.pyplot as plt
from mlflow import MlflowClient
from mlflow.models import infer_signature
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

## Passo 1 — Configurar o tracking
Um banco SQLite local guarda os runs e o model registry.

In [ ]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("deteccao-de-fraude")
print(mlflow.get_tracking_uri())

## Passo 2 — Dados com versão
O problema de fraude da Semana 1. O hash dos dados identifica exatamente o que foi usado.

In [ ]:
SEMENTE = 42
X, y = make_classification(n_samples=30_000, n_features=20, n_informative=6, n_redundant=4, weights=[0.995],
                           class_sep=1.3, flip_y=0.001, random_state=SEMENTE)
X = pd.DataFrame(X, columns=[f"f{i}" for i in range(20)])
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEMENTE)
hash_dados = hashlib.sha256(pd.util.hash_pandas_object(X, index=True).values.tobytes() + y.tobytes()).hexdigest()[:12]
print("hash dos dados:", hash_dados)

## Passo 3 — Um run completo
Dentro de `mlflow.start_run`: registre parâmetros, treine, registre métricas, a figura da confusion matrix e o modelo com signature.

In [ ]:
def custo_negocio(y_true, y_pred, c_fn=2000, c_fp=15):
    vn, fp, fn, vp = confusion_matrix(y_true, y_pred).ravel(); return float(fn * c_fn + fp * c_fp)

def executar(params, limiar=0.5, nome=None):
    with mlflow.start_run(run_name=nome) as run:
        mlflow.log_params({**params, "limiar": limiar, "semente": SEMENTE, "hash_dados": hash_dados})
        mlflow.set_tags({"autor": "eu", "motivo": "comparação de configurações"})
        modelo = Pipeline([("scaler", StandardScaler()), ("clf", RandomForestClassifier(random_state=SEMENTE, n_jobs=-1, **params))]).fit(X_tr, y_tr)
        pred = (modelo.predict_proba(X_te)[:, 1] >= limiar).astype(int)
        metricas = {"precisao": precision_score(y_te, pred, zero_division=0), "recall": recall_score(y_te, pred),
                    "f1": f1_score(y_te, pred), "custo": custo_negocio(y_te, pred)}
        mlflow.log_metrics(metricas)
        fig = ConfusionMatrixDisplay.from_predictions(y_te, pred, cmap="Blues").figure_
        mlflow.log_figure(fig, "matriz_confusao.png")
        plt.close(fig)
        mlflow.sklearn.log_model(modelo, name="modelo", serialization_format="cloudpickle", signature=infer_signature(X_te, modelo.predict(X_te)), input_example=X_te.head(3))
        return run.info.run_id, metricas

rid, m = executar({"n_estimators": 100}, nome="base"); print(rid, m)

## Passo 4 — Vários runs

In [ ]:
grade = [({"n_estimators": 100, "class_weight": "balanced_subsample", "min_samples_leaf": 5}, 0.5, "balanceado"),
         ({"n_estimators": 100, "class_weight": "balanced_subsample", "min_samples_leaf": 5}, 0.2, "balanceado-limiar-0.2"),
         ({"n_estimators": 300, "class_weight": "balanced_subsample", "min_samples_leaf": 3}, 0.2, "grande-limiar-0.2")]
for params, limiar, nome in grade:
    print(nome, executar(params, limiar, nome)[1])

## Passo 5 — Comparar pelo critério de negócio
Use `mlflow.search_runs` para listar os runs ordenados pelo menor custo.

In [ ]:
runs = mlflow.search_runs(order_by=["metrics.custo ASC"])
runs[["tags.mlflow.runName", "params.limiar", "metrics.recall", "metrics.precisao", "metrics.custo", "run_id"]]

## Passo 6 — Registrar a melhor versão
Registre o modelo do melhor run com o nome `detector-fraude` e dê a ele o alias `champion`.

In [ ]:
melhor = runs.iloc[0]["run_id"]; cliente = MlflowClient()
versao = mlflow.register_model(f"runs:/{melhor}/modelo", "detector-fraude")
cliente.set_registered_model_alias("detector-fraude", "champion", versao.version)
print("versão registrada:", versao.version)

## Passo 7 — Carregar por alias
O código de inferência não conhece números de versão.

In [ ]:
em_producao = mlflow.pyfunc.load_model("models:/detector-fraude@champion")
print(em_producao.predict(X_te.head(5)))

## Passo 8 — Desafiante, promoção e reversão
Registre o segundo melhor run como nova versão com alias `challenger`. Depois promova-o a `champion` e, em seguida, reverta.

In [ ]:
segunda = runs.iloc[1]["run_id"]
v2 = mlflow.register_model(f"runs:/{segunda}/modelo", "detector-fraude")
cliente.set_registered_model_alias("detector-fraude", "challenger", v2.version)
def campeao(): return cliente.get_model_version_by_alias("detector-fraude", "champion").version
print("champion antes:", campeao())
cliente.set_registered_model_alias("detector-fraude", "champion", v2.version)
print("champion após promoção:", campeao())
cliente.set_registered_model_alias("detector-fraude", "champion", versao.version)
print("champion após reversão:", campeao())

## Passo 9 — Da versão de volta à origem
Dada a versão em produção, recupere o run que a gerou, seus parâmetros e o hash dos dados. É a pergunta de auditoria.

In [ ]:
mv = cliente.get_model_version_by_alias("detector-fraude", "champion")
origem = cliente.get_run(mv.run_id)
print("parâmetros:", origem.data.params); print("métricas:", {k: round(v, 3) for k, v in origem.data.metrics.items()})

## Passo 10 (opcional) — Interface web
No Colab, a interface do MLflow pode ser aberta por um túnel de porta. Localmente: `mlflow ui --backend-store-uri sqlite:///mlflow.db`.